# Creating a Simple Agent with Tracing

In [21]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [22]:
from agents import Agent, Runner, function_tool, trace
from openai.types.responses import ResponseTextDeltaEvent

Create calculator and EVS lookup tools for the tutors:

In [23]:
def calculate_math(operation: str, first_number: float, second_number: float) -> str:
    """Calculate addition, subtraction, multiplication, or division."""
    operation = operation.strip().lower()
    if operation == "add":
        result = first_number + second_number
        symbol = "+"
    elif operation == "subtract":
        result = first_number - second_number
        symbol = "-"
    elif operation == "multiply":
        result = first_number * second_number
        symbol = "*"
    elif operation == "divide":
        if second_number == 0:
            raise ValueError("Cannot divide by zero.")
        result = first_number / second_number
        symbol = "/"
    else:
        raise ValueError("operation must be add, subtract, multiply, or divide.")

    return f"{first_number:g} {symbol} {second_number:g} = {result:g}"


def lookup_evs_concept(concept: str) -> str:
    """Look up a basic explanation of an environmental science concept."""
    concept_data = {
        "photosynthesis": (
            "Plants use sunlight, water, and carbon dioxide to make glucose and release oxygen."
        ),
        "ozone depletion": (
            "The thinning of the ozone layer, mainly caused by chemicals such as CFCs, "
            "allows more harmful ultraviolet radiation to reach Earth."
        ),
        "global warming": (
            "The long-term rise in Earth's average temperature, primarily caused by "
            "greenhouse gases released through human activities."
        ),
        "greenhouse effect": (
            "Atmospheric gases trap some of Earth's heat. This natural process keeps "
            "Earth warm, but extra greenhouse gases strengthen it."
        ),
    }

    concept_key = concept.strip().lower()
    if concept_key in concept_data:
        return f"{concept_key.title()}: {concept_data[concept_key]}"

    available_concepts = ", ".join(concept_data)
    return f"I don't have information about '{concept}'. Try one of these: {available_concepts}."


math_tool = function_tool(calculate_math)
evs_lookup_tool = function_tool(lookup_evs_concept)

Create a simple Math Tutor Agent

In [24]:
math_tutor = Agent(
    name="Math Tutor",
    instructions="""
    You are a patient math tutor.
    Explain the steps clearly and use language appropriate for a student.
    Use the calculator tool for arithmetic calculations.
    """,
    tools=[math_tool],
)

Create a simple Environmental Studies (EVS) Assistant Agent:

In [25]:
evs_agent = Agent(
    name="EVS Assistant",
    instructions="""
    You are a helpful Environmental Studies tutor.
    Use the EVS lookup tool for questions about photosynthesis, ozone depletion,
    global warming, and the greenhouse effect. Explain concepts clearly for a student.
    """,
    tools=[evs_lookup_tool],
)

Let's execute the Agent:

In [26]:
with trace("Simple Math Tutor"):
    result = await Runner.run(math_tutor, "How do I solve 3x + 5 = 20?")

print(result)

RunResult:
- Last agent: Agent(name="Math Tutor", ...)
- Final output (str):
    Steps:
    1) Subtract 5 from both sides: 3x + 5 = 20 → 3x = 20 − 5 = 15
    2) Divide both sides by 3: 3x = 15 → x = 15 ÷ 3 = 5
    
    Check: 3(5) + 5 = 15 + 5 = 20, which matches.
    
    Answer: x = 5
- 9 new item(s)
- 4 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Try the EVS Assistant:

In [27]:
with trace("EVS Assistant"):
    result = await Runner.run(evs_agent, "What is photosynthesis?")

print(result.final_output)

Photosynthesis is the process by which plants (and some algae and bacteria) use sunlight to convert water and carbon dioxide into glucose (a sugar) and oxygen.

Key points:
- Where it happens: in chloroplasts of plant cells (contain chlorophyll, the green pigment).
- Basic inputs: light energy, water (H2O), and carbon dioxide (CO2).
- Basic outputs: glucose (C6H12O6) and oxygen (O2).
- Overall equation (simplified): 6 CO2 + 6 H2O + light energy → C6H12O6 + 6 O2
- It happens in two stages:
  1) Light-dependent reactions (in the thylakoid membranes): capture light to make energy-carrying molecules (ATP and NADPH) and release O2.
  2) Calvin cycle (light-independent reactions): use ATP and NADPH to convert CO2 into glucose.
- Why it matters: provides the oxygen we breathe and forms the base of nearly all food webs.


Streaming the answer to the screen, token by token

In [28]:
response_stream = Runner.run_streamed(math_tutor, "What is 12 multiplied by 8?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

96

Explanation (optional): 12 × 8 = (10 + 2) × 8 = 10×8 + 2×8 = 80 + 16 = 96.

_Good Job!_